# 09b — Bounded queue pressure

Occupancy and offered, accepted, processed, and drained rates are separate bounded-channel observations. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

try: batch=resolve_result_batch('e-backpressure', diagnostic_path=os.environ.get('E_BACKPRESSURE_DIR'))
except (FileNotFoundError, RuntimeError, ValueError): batch=None
rows=[] if batch is None else [value for _,value in passed_json(batch,'backpressure.json')]
if rows:
    df=pd.DataFrame([{'classification':r['classification'],'peak_occupancy':r['peak_occupancy'],'offered_msg_s':r['rates_msg_s']['offered'],'accepted_msg_s':r['rates_msg_s']['accepted'],'processed_msg_s':r['rates_msg_s']['processed'],'drained_msg_s':r['rates_msg_s']['drained'],'rss_within_limit':r['memory']['within_limit']} for r in rows])
    print(evidence_label(len(df), 'ratio, messages/second, bytes', False)); display(df)
else:
    display(pd.DataFrame([pending_record('bounded queue pressure','no passed backpressure.json leaf','ratio, messages/second, bytes')]))
